# Notebook 06 — AI-Generated Executive Narrative

**Tujuan:** Demonstrasi peran **Prompt Architect** (lihat CLAUDE.md Section 2) — auto-generate narasi executive dari metrics A/B test pakai **Google Gemini API (gratis)**.

**Mengapa LLM, bukan template string biasa?**
- Template string kaku, sulit nuansa (mis. "effect kecil tapi signifikan dgn n besar" butuh framing kontekstual).
- LLM bisa contextualize ke konteks bisnis Indonesia, generate caveat, dan adapt format ke audiens.
- **Tapi:** rentan halusinasi statistik kalau prompt tidak ketat. Itu sebabnya `prompts/exec_narrative.py` berisi STATISTICAL SAFEGUARDS yang explicit.

**Pipeline:**
```
df_clean.parquet → compute metrics dict → build_user_prompt() → Gemini API → markdown narrative
```

**Output:** `output/ai_narrative.md` (committed to repo sebagai demo artifact).

**Cost:** **GRATIS** dengan Gemini 2.0 Flash free tier (15 req/menit, 1500 req/hari, 1M token/hari). Cukup untuk demo + 100+ regenerate tanpa bayar sepeser pun.

**Prerequisites:**
1. `pip install google-generativeai python-dotenv` (sudah ada di `environment.yml`)
2. Dapat API key gratis: https://aistudio.google.com/app/apikey → login Google → "Create API Key" (no credit card)
3. Copy `.env.example` ke `.env` di root repo, isi `GOOGLE_API_KEY`
4. Notebook 01 sudah pernah di-run (butuh `output/df_clean.parquet`)

**Note arsitektur:** Prompt template di `prompts/exec_narrative.py` **portable** — kalau nanti mau pindah ke Claude/GPT/Llama, cuma cell API call yang ganti. System prompt tidak.

## 0. Setup: Imports & Client

In [1]:
import json
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from scipy.stats import ttest_ind
from statsmodels.stats.proportion import proportions_ztest

# .env ada di root repo (2 level di atas notebook ini)
ROOT = Path('../..').resolve()
load_dotenv(ROOT / '.env')

api_key = os.getenv('OPENROUTER_API_KEY')
if not api_key:
    raise RuntimeError(
        f'OPENROUTER_API_KEY tidak ditemukan. Pastikan {ROOT / ".env"} ada & terisi.\n'
        f'Lihat .env.example untuk format. Dapat key gratis di https://openrouter.ai/keys'
    )

client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=api_key,
)

MODEL = 'openai/gpt-oss-120b:free'
OUT = Path('../output')

# Import prompt module (folder notebooks/prompts/)
sys.path.insert(0, str(Path('prompts').resolve()))
from exec_narrative import SYSTEM_PROMPT, build_user_prompt

print(f'API ready. Model: {MODEL}')
print(f'System prompt length: {len(SYSTEM_PROMPT):,} chars (~{len(SYSTEM_PROMPT)//4} tokens)')

API ready. Model: openai/gpt-oss-120b:free
System prompt length: 3,254 chars (~813 tokens)


## 1. Load A/B Test Metrics

Recompute metrics dari `df_clean.parquet` untuk transparency — kita bisa lihat persis angka apa yang dikirim ke LLM. Pakai metrics dari notebook 02 (conversion test) + notebook 03 (revenue test).

In [2]:
df = pd.read_parquet(OUT / 'df_clean.parquet')

ctrl = df[df['group'] == 'control']
treat = df[df['group'] == 'treatment']

# Conversion rate test (z-test for proportions)
conv_ctrl = ctrl['converted'].sum()
conv_treat = treat['converted'].sum()
p_ctrl = conv_ctrl / len(ctrl)
p_treat = conv_treat / len(treat)
z_stat, p_conv = proportions_ztest(
    [conv_treat, conv_ctrl], [len(treat), len(ctrl)]
)
lift_pct = (p_treat - p_ctrl) / p_ctrl * 100

# Revenue test (Welch t-test) + bootstrap CI
rev_ctrl = ctrl['revenue'].values
rev_treat = treat['revenue'].values
t_stat, p_rev = ttest_ind(rev_treat, rev_ctrl, equal_var=False)
pooled_std = np.sqrt((rev_ctrl.std() ** 2 + rev_treat.std() ** 2) / 2)
cohens_d = (rev_treat.mean() - rev_ctrl.mean()) / pooled_std

np.random.seed(42)
N_BOOT = 10_000
boot_diffs = np.array([
    np.mean(np.random.choice(rev_treat, len(rev_treat), replace=True))
    - np.mean(np.random.choice(rev_ctrl, len(rev_ctrl), replace=True))
    for _ in range(N_BOOT)
])
ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])

# Business projection
daily_traffic = df.groupby(df['timestamp'].dt.date)['user_id'].count().mean()
mean_diff = rev_treat.mean() - rev_ctrl.mean()
annual_uplift = mean_diff * daily_traffic * 365

metrics = {
    'test_metadata': {
        'test_duration_days': int((df['timestamp'].max() - df['timestamp'].min()).days),
        'n_control': int(len(ctrl)),
        'n_treatment': int(len(treat)),
        'daily_traffic_avg': int(daily_traffic),
    },
    'conversion_test': {
        'method': 'z-test for proportions',
        'conversion_rate_control': round(float(p_ctrl), 4),
        'conversion_rate_treatment': round(float(p_treat), 4),
        'relative_lift_pct': round(float(lift_pct), 2),
        'z_statistic': round(float(z_stat), 4),
        'p_value': round(float(p_conv), 4),
        'significant_at_005': bool(p_conv < 0.05),
    },
    'revenue_test': {
        'method': "Welch's t-test (unequal variance) + bootstrap CI (10k iter)",
        'mean_revenue_control_usd': round(float(rev_ctrl.mean()), 4),
        'mean_revenue_treatment_usd': round(float(rev_treat.mean()), 4),
        'mean_diff_usd': round(float(mean_diff), 4),
        't_statistic': round(float(t_stat), 4),
        'p_value': round(float(p_rev), 4),
        'cohens_d': round(float(cohens_d), 4),
        'bootstrap_ci_95_lower_usd': round(float(ci_low), 4),
        'bootstrap_ci_95_upper_usd': round(float(ci_high), 4),
        'ci_contains_zero': bool(ci_low < 0 < ci_high),
        'significant_at_005': bool(p_rev < 0.05),
    },
    'business_projection': {
        'annual_revenue_uplift_usd': round(float(annual_uplift), 0),
        'note': 'Projection naif: mean_diff x daily_traffic x 365. Tidak akun seasonality.',
    },
}

print(json.dumps(metrics, indent=2))

{
  "test_metadata": {
    "test_duration_days": 21,
    "n_control": 145274,
    "n_treatment": 145310,
    "daily_traffic_avg": 12634
  },
  "conversion_test": {
    "method": "z-test for proportions",
    "conversion_rate_control": 0.1204,
    "conversion_rate_treatment": 0.1188,
    "relative_lift_pct": -1.31,
    "z_statistic": -1.3109,
    "p_value": 0.1899,
    "significant_at_005": false
  },
  "revenue_test": {
    "method": "Welch's t-test (unequal variance) + bootstrap CI (10k iter)",
    "mean_revenue_control_usd": 5.3962,
    "mean_revenue_treatment_usd": 5.3504,
    "mean_diff_usd": -0.0458,
    "t_statistic": -0.7975,
    "p_value": 0.4252,
    "cohens_d": -0.003,
    "bootstrap_ci_95_lower_usd": -0.1564,
    "bootstrap_ci_95_upper_usd": 0.0687,
    "ci_contains_zero": true,
    "significant_at_005": false
  },
  "business_projection": {
    "annual_revenue_uplift_usd": -211093.0,
    "note": "Projection naif: mean_diff x daily_traffic x 365. Tidak akun seasonality."
  }

## 2. Build Prompt

System prompt (cached) berisi role + output format + STATISTICAL SAFEGUARDS. User prompt berisi metrics JSON + business context.

In [3]:
BUSINESS_CONTEXT = '''
Test ini menguji desain ulang product page (Treatment) vs versi existing (Control).
Hipotesis tim Marketing: design baru lebih jelas CTA, expect lift konversi ~5% & revenue per user +$0.10.
Durasi test 23 hari, sample acak 50/50 split. Tidak ada perubahan price/promo selama test.
'''.strip()

user_prompt = build_user_prompt(
    test_name='New Product Page A/B Test',
    metrics=metrics,
    business_context=BUSINESS_CONTEXT,
)

print('--- USER PROMPT (preview) ---')
print(user_prompt[:500])
print(f'\n... ({len(user_prompt):,} chars total)')

--- USER PROMPT (preview) ---
# A/B Test: New Product Page A/B Test

## Business Context
Test ini menguji desain ulang product page (Treatment) vs versi existing (Control).
Hipotesis tim Marketing: design baru lebih jelas CTA, expect lift konversi ~5% & revenue per user +$0.10.
Durasi test 23 hari, sample acak 50/50 split. Tidak ada perubahan price/promo selama test.

## Test Metrics (sumber: hasil notebook analisis)

```json
{
  "test_metadata": {
    "test_duration_days": 21,
    "n_control": 145274,
    "n_treatment": 145

... (1,610 chars total)


## 3. Call Gemini API (dengan fallback chain)

Konfigurasi:
- **Model preference:** `gemini-2.5-flash` → `gemini-2.0-flash` → `gemini-1.5-flash` (otomatis fallback kalau model tertentu dapat `limit: 0` di project/region lo)
- **System instruction:** dipasang ke model
- **Temperature 0.3:** lower variance untuk narasi statistik
- **max_output_tokens:** 2000

**Kenapa fallback chain?**
Free tier Gemini berbeda per region & per model. Project Google Cloud di Indonesia, atau project yang sudah linked ke billing, kadang dapat `limit: 0` untuk model tertentu. Fallback otomatis = notebook ini robust di banyak setup tanpa edit kode manual.

In [4]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_prompt},
    ],
    temperature=0.3,
    max_tokens=3000,
)

finish = response.choices[0].finish_reason
print(f'Model used    : {MODEL}')
print(f'Finish reason : {finish}')
if finish == 'length':
    print('⚠️  Output terpotong (max_tokens tercapai) — naikkan max_tokens kalau perlu')
print(f'Output preview: {response.choices[0].message.content[:200]}...')

Model used    : openai/gpt-oss-120b:free
Finish reason : stop
Output preview: ## TL;DR
**NO‑GO ❌** – Desain ulang halaman produk tidak meningkatkan konversi maupun revenue; hasilnya bahkan sedikit menurun dengan bukti statistik yang tidak signifikan.

## Key Metrics

| Metrik |...


## 4. Extract, Display, & Save Narrative

In [5]:
from IPython.display import Markdown, display

narrative = response.choices[0].message.content

out_path = OUT / 'ai_narrative.md'
out_path.write_text(narrative, encoding='utf-8')
print(f'Saved: {out_path} ({len(narrative):,} chars)')

display(Markdown(narrative))

Saved: ..\output\ai_narrative.md (2,942 chars)


## TL;DR
**NO‑GO ❌** – Desain ulang halaman produk tidak meningkatkan konversi maupun revenue; hasilnya bahkan sedikit menurun dengan bukti statistik yang tidak signifikan.

## Key Metrics

| Metrik | Control | Treatment | Selisih |
|--------|---------|-----------|---------|
| Conversion Rate | 12.04 % | 11.88 % | **‑0.16 ppt** (‑1.31 % rel.) |
| Mean Revenue per User (USD) | 5.3962 | 5.3504 | **‑0.0458** |
| p‑value (conversion) | 0.1899 | – | > 0.05 |
| p‑value (revenue) | 0.4252 | – | > 0.05 |
| Cohen’s d (revenue) | – | – | **‑0.003** (sangat kecil) |
| 95 % Bootstrap CI (revenue diff) | – | – | [‑0.1564, 0.0687] |

## Statistical Interpretation
- **Conversion:** p = 0.19 > 0.05 → gagal menolak H₀. Tidak ada bukti yang cukup kuat bahwa desain baru mengubah rasio konversi. Z‑statistik –1.31 menunjukkan arah penurunan, tetapi tidak signifikan.  
- **Revenue:** p = 0.425 > 0.05 → gagal menolak H₀. Selisih rata‑rata –0.0458 USD tidak signifikan; interval kepercayaan 95 % mencakup nol (‑0.1564 – 0.0687), artinya estimasi tidak presisi. Cohen’s d = ‑0.003 (< 0.2) menandakan efek praktis yang sangat kecil.  
- Karena sampel besar (≈ 145 k per grup) dan efek sangat kecil, dapat disimpulkan bahwa **efek nyata kemungkinan sangat dekat nol**.

## Business Decision
**NO‑GO ❌**  
Tidak ada bukti statistik maupun praktis bahwa redesign halaman produk meningkatkan konversi atau revenue. Bahkan, estimasi menunjukkan penurunan marginal yang tidak signifikan secara statistik, sehingga mengimplementasikan perubahan ini akan berisiko menurunkan pendapatan tahunan sekitar **USD ‑211 k** (proyeksi naïf).

## Recommended Actions
1. **Retain** versi halaman produk yang ada (control) untuk semua segmen.
2. **Investigate** penyebab penurunan kecil pada conversion (mis. elemen CTA, layout mobile) dengan studi kualitatif atau tes A/B yang lebih tersegmentasi.
3. **Iterate** desain baru secara bertahap—misalnya, uji satu elemen (warna tombol, teks CTA) dalam eksperimen terpisah sebelum melakukan redesign penuh.

## Caveats & Limitations
- Tidak dapat disimpulkan bahwa redesign akan berdampak pada metrik jangka panjang (retention, LTV) karena percobaan hanya 21 hari.
- Asumsi traffic dan perilaku pengguna tetap konstan; tidak memperhitungkan fluktuasi musiman (mis. Harbolnas).
- Proyeksi revenue tahunan bersifat naïf; tidak mengakomodasi faktor musiman, promosi, atau perubahan perilaku pengguna di masa depan.

> **Relevansi Indonesia:** Pada marketplace berskala Tokopedia/Shopee, traffic mobile‑first sangat tinggi dan keputusan desain halaman produk mempengaruhi jutaan sesi harian. Mengingat volume traffic (≈ 12.6 k hari‑hari) dan nilai rata‑rata order, bahkan perubahan kecil pada conversion atau revenue per user dapat berdampak signifikan pada margin. Oleh karena itu, keputusan untuk **tidak meluncurkan redesign** kini melindungi margin dan menghindari potensi kerugian pada periode belanja besar seperti Harbolnas.

## 5. Usage & Free Tier Status

Cek token usage untuk track free tier consumption.

**Gemini 2.0 Flash free tier:**
- 15 requests / menit
- 1,500 requests / hari
- 1,000,000 tokens / hari

Untuk konteks: 1 generation di notebook ini ~3000 token total. Free tier ~ **300+ generation/hari** — lebih dari cukup untuk pakai sehari-hari + demo ke recruiter.

In [6]:
u = response.usage

print('=== TOKEN USAGE ===')
print(f'Model used             : {MODEL}')
print(f'Prompt (system + user) : {u.prompt_tokens:,} tokens')
print(f'Output (narrative)     : {u.completion_tokens:,} tokens')
print(f'TOTAL                  : {u.total_tokens:,} tokens')
print()
print('=== COST ===')
print(f'$0.00 — OpenRouter free tier ({MODEL})')
print()
print('=== FREE TIER STATUS ===')
print(f'Per call ~{u.total_tokens:,} tokens')
print(f'OpenRouter :free models — rate limited per minute, no daily token cap')

=== TOKEN USAGE ===
Model used             : openai/gpt-oss-120b:free
Prompt (system + user) : 1,512 tokens
Output (narrative)     : 888 tokens
TOTAL                  : 2,400 tokens

=== COST ===
$0.00 — OpenRouter free tier (openai/gpt-oss-120b:free)

=== FREE TIER STATUS ===
Per call ~2,400 tokens
OpenRouter :free models — rate limited per minute, no daily token cap


## Executive Summary

| Aspek | Hasil |
|-------|-------|
| Demonstrasi peran | **Prompt Architect** (CLAUDE.md Section 2) |
| LLM provider | Google Gemini 2.0 Flash (free tier) |
| Pipeline | Metrics dict → structured prompt → API call → markdown narrative |
| Safeguards | STATISTICAL SAFEGUARDS di system prompt (p-value, CI, effect size correctness) |
| Re-usability | `exec_narrative.py` portable — bisa swap ke Claude/GPT/Llama tanpa ubah prompt |
| Cost | **$0.00** (Gemini free tier — 1500 calls/hari, 1M token/hari) |
| Output | `output/ai_narrative.md` (committed sbg portfolio artifact) |

**Business Recommendation:**
1. **Extend** pipeline ini ke 4 proyek lain (sales, churn, cohort, supply-chain) untuk demonstrasi konsistensi.
2. **Validate** output narrative dengan Ray (mahasiswa statistika) — terutama klaim p-value & CI interpretation. LLM kadang slip di interpretasi frequentist vs Bayesian.
3. **Document** prompt design decisions — sudah didocument di docstring `exec_narrative.py`.

**Why Gemini (alasan teknis untuk recruiter):**
- Untuk task narrative dari structured input (metrics JSON → markdown), Gemini 2.0 Flash quality ≈ Claude Sonnet, tapi gratis.
- Architecture pipeline tetap portable: swap SDK 5 baris, prompt tidak diubah.
- Demonstrasi pragmatisme: pick the right tool for the budget, bukan default ke yang paling mahal.

> **Relevansi Indonesia:** Tokopedia/Shopee Indonesia rutin lakukan ratusan A/B test simultan; bottleneck biasanya bukan compute tapi *interpretation* hasil oleh non-data team. Auto-narrative pipeline seperti ini bisa kurangi friction dari analyst → PM dari 2-3 hari menjadi same-day — dan dengan Gemini free tier, scale ke 1000+ test/bulan tanpa cost concern.